# Text Classification with TF-IDF and Logistic Regression

This notebook demonstrates a complete multiclass text-classification workflow using a small self-contained dataset.

The task is to classify short documents into three topics:

- **finance**
- **sports**
- **technology**

No external dataset download is required.

## Learning Objectives

After completing this notebook, students should be able to:

1. organize labeled text data in a tabular structure;
2. split data into training and test subsets;
3. convert raw text into TF-IDF features;
4. train a multiclass logistic-regression classifier;
5. evaluate predictions using accuracy, precision, recall, F1 score, and a confusion matrix;
6. inspect influential terms for each class;
7. classify new, unseen text;
8. explain the limitations of a small demonstration dataset.

## 1. Import the required libraries

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

## 2. Create a small labeled text dataset

Each document is assigned one topic label. The examples are intentionally short so the full workflow remains easy to inspect.

In [ ]:
documents = [
    # Finance
    ("The bank approved the business loan application.", "finance"),
    ("Investors reacted to the quarterly earnings report.", "finance"),
    ("The central bank increased interest rates.", "finance"),
    ("The company issued new shares to raise capital.", "finance"),
    ("Inflation reduced household purchasing power.", "finance"),
    ("The stock market closed higher after the announcement.", "finance"),
    ("The customer deposited money into a savings account.", "finance"),
    ("The finance team prepared the annual budget.", "finance"),
    ("Bond prices changed after the policy decision.", "finance"),
    ("The investor diversified the portfolio to reduce risk.", "finance"),
    ("The firm reported strong revenue growth.", "finance"),
    ("Currency exchange rates moved during trading.", "finance"),

    # Sports
    ("The football team won the championship match.", "sports"),
    ("The player scored two goals in the final.", "sports"),
    ("The coach changed the defensive formation.", "sports"),
    ("The tennis player advanced to the semifinal.", "sports"),
    ("The basketball game ended in overtime.", "sports"),
    ("The athlete broke the national record.", "sports"),
    ("The referee stopped the match after the injury.", "sports"),
    ("The club signed a new striker.", "sports"),
    ("The team practiced before the tournament.", "sports"),
    ("The goalkeeper saved the penalty kick.", "sports"),
    ("The runner completed the marathon.", "sports"),
    ("The league published the season schedule.", "sports"),

    # Technology
    ("The software update fixed several security issues.", "technology"),
    ("The developer trained a machine learning model.", "technology"),
    ("The company launched a new mobile application.", "technology"),
    ("The server processed requests through the cloud platform.", "technology"),
    ("The cybersecurity team detected a network attack.", "technology"),
    ("The database stored millions of user records.", "technology"),
    ("The programmer improved the Python code.", "technology"),
    ("The computer uses a faster processor.", "technology"),
    ("The robot navigated using artificial intelligence.", "technology"),
    ("The research team tested a new algorithm.", "technology"),
    ("The application encrypts sensitive information.", "technology"),
    ("The engineers deployed the model to production.", "technology"),
]

data = pd.DataFrame(
    documents,
    columns=["text", "label"],
)

print("Dataset shape:", data.shape)
display(data.head())

print("\nClass distribution:")
display(
    data["label"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

## 3. Split the dataset

A stratified split preserves the class proportions in both the training and test subsets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.25,
    random_state=42,
    stratify=data["label"],
)

print("Training documents:", len(X_train))
print("Test documents:", len(X_test))

print("\nTraining class counts:")
print(y_train.value_counts().sort_index())

print("\nTest class counts:")
print(y_test.value_counts().sort_index())

## 4. Build the classification pipeline

The pipeline contains two stages.

### TF-IDF Vectorization

`TfidfVectorizer` converts text into numerical features. Terms receive higher weight when they are frequent in one document but less common across the complete corpus.

### Logistic Regression

Logistic regression learns one decision function for each class and estimates which topic best matches a document.

Using a pipeline ensures that vectorization is fitted only on the training data.

In [ ]:
classifier = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=1,
            ),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1_000,
                random_state=42,
            ),
        ),
    ]
)

classifier.fit(X_train, y_train)

## 5. Evaluate the classifier

In [ ]:
predictions = classifier.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions,
)

print(f"Accuracy: {accuracy:.2%}")
print()

print(
    classification_report(
        y_test,
        predictions,
        zero_division=0,
    )
)

## 6. Display the confusion matrix

The confusion matrix shows how many documents from each true class were assigned to each predicted class.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    labels=classifier.classes_,
    display_labels=classifier.classes_,
)

plt.title("Text Classification Confusion Matrix")
plt.tight_layout()
plt.show()

## 7. Inspect influential terms

For multiclass logistic regression, each class has a coefficient for every TF-IDF feature.

Large positive coefficients indicate terms that strongly support a class prediction.

In [ ]:
vectorizer = classifier.named_steps["tfidf"]
model = classifier.named_steps["model"]

feature_names = vectorizer.get_feature_names_out()
top_term_count = 10

for class_index, class_name in enumerate(model.classes_):
    class_coefficients = model.coef_[class_index]

    top_indices = class_coefficients.argsort()[
        -top_term_count:
    ][::-1]

    top_terms = feature_names[top_indices]

    print(f"{class_name}:")
    print(", ".join(top_terms))
    print()

## 8. Classify new documents

The trained pipeline applies the same TF-IDF transformation before generating predictions and class probabilities.

In [ ]:
new_documents = [
    "The company released a new cloud security platform.",
    "The striker scored the winning goal.",
    "The bank announced lower borrowing costs.",
]

new_predictions = classifier.predict(
    new_documents
)

new_probabilities = classifier.predict_proba(
    new_documents
)

for text, prediction, probabilities in zip(
    new_documents,
    new_predictions,
    new_probabilities,
):
    probability_by_class = {
        class_name: round(float(probability), 4)
        for class_name, probability in zip(
            classifier.classes_,
            probabilities,
        )
    }

    print("Text:", text)
    print("Predicted class:", prediction)
    print("Probabilities:", probability_by_class)
    print()

## Result Interpretation

The model learns associations between terms and topic labels.

Examples include:

- financial terms such as `bank`, `loan`, `market`, and `revenue`;
- sports terms such as `team`, `goal`, `match`, and `player`;
- technology terms such as `software`, `model`, `server`, and `application`.

The classifier can generalize to new sentences when they contain patterns similar to those observed during training.

## Evaluation Metrics

The classification report includes:

- **precision:** how often predictions for a class are correct;
- **recall:** how many actual examples of a class are detected;
- **F1 score:** harmonic mean of precision and recall;
- **support:** number of test examples for each class.

Accuracy reports the overall proportion of correct predictions.

## Limitations

This notebook is a compact teaching example.

Its limitations include:

- a very small manually constructed dataset;
- short and strongly topic-specific sentences;
- limited linguistic variation;
- no cross-validation;
- no hyperparameter optimization;
- no class imbalance;
- no ambiguous or multilingual documents;
- no independent validation corpus.

High performance on this dataset should not be interpreted as evidence of production-level generalization.

## Exercises

1. Add more examples to every class.
2. Introduce ambiguous documents.
3. Compare unigram and bigram features.
4. Replace logistic regression with Naive Bayes.
5. Evaluate the model with stratified cross-validation.
6. Add a fourth topic.
7. Save and reload the trained pipeline with `joblib`.
8. Test the model on longer documents.
9. Examine incorrectly classified examples.
10. Compare TF-IDF with pretrained text embeddings.